# Using the EnvisionHGdetector Package
We just follow this tutorial here: https://envisionbox.org/embedded_UsingEnvisionHGdetector_package.html

In [4]:
import os
import glob
from IPython.display import Video

# Define folders
videofoldertoday = ['../../videos/']
outputfolder = '../../gestureclassifications/'

# Create output directory
os.makedirs(outputfolder, exist_ok=True)

# List available videos across all folders
videos = []
for folder in videofoldertoday:
    print(os.path.abspath(folder))
    # loop through each subfolder
    for subfolder in os.listdir(folder):
        for subsubfolder in os.listdir(os.path.join(folder, subfolder)):
            #print(f"Searching in {os.path.join(folder, subfolder, subsubfolder)}")
            foldertocheck = os.path.join(folder, subfolder, subsubfolder)
            # Find all matching mp4s in the current folder
            all_found = glob.glob(os.path.join(foldertocheck, '*cam02.mp4'))           
            # Filter out OS metadata files
            # Filter updates:
            # 1. Must NOT start with '._'
            # 2. Must exist and have a file size greater than 0 bytes
            valid_files = [
                f for f in all_found 
                if not os.path.basename(f).startswith('._') 
                and os.path.exists(f) 
                and os.path.getsize(f) > 0
            ]
            videos.extend(valid_files)

print(f"Found {len(videos)} valid videos to process")



d:\Research_projects\TilburgMultiscaleSummerschool2026\Datasets\BalanceCorpus\videos
Found 188 valid videos to process


In [ ]:
import tempfile
import pathlib
import os
from envisionhgdetector import GestureDetector
import os
# Create detector with combined model
detector = GestureDetector(
    model_type="combined",
    cnn_motion_threshold=0.9,    # Motion gate sensitivity
    cnn_gesture_threshold=0.9,   # CNN gesture confidence
    lgbm_threshold=0.95,          # LightGBM gesture probability
    min_gap_s=0.0,               # Merge gaps smaller than this
    min_length_s=0.1             # Minimum gesture duration
)

# Find the common drive or root of your first video folder to host the temp directory
# This ensures the temp folder is created on the 'G:' drive
video_root_dir = os.path.dirname(os.path.abspath(videos[0]))

# 1. Create a temporary directory explicitly on the G: drive
with tempfile.TemporaryDirectory(dir=video_root_dir) as tmpdir:
    tmp_path = pathlib.Path(tmpdir)
    
    # 2. Populate it with hard links
    for video_path in videos:  
        video_file = pathlib.Path(video_path).resolve()
        dest_path = tmp_path / video_file.name
        
        # make sure the disk assigned is ok
        os.link(str(video_file), str(dest_path))
    
    # 3. Pass the temporary folder to your detector
    print(f"Step 1: Processing {len(videos)} videos via cross-drive safe hardlink layer...")
    detector.process_folder(
        input_folder=str(tmp_path),
        output_folder=outputfolder,
    )
    
    


✓ CNN model loaded from d:\Programs\Conda_packages\envs\envision\lib\site-packages\envisionhgdetector\model\cnn_default.h5
✓ LightGBM model loaded from d:\Programs\Conda_packages\envs\envision\lib\site-packages\envisionhgdetector\model\lightgbm_default.pkl

✓ Combined model initialized
  CNN thresholds: motion=0.9, gesture=0.9
  LightGBM threshold: 0.95
  CNN ready: True
  LightGBM ready: True
Initialized Combined (CNN+LightGBM) gesture detector
  CNN thresholds: motion=0.9, gesture=0.9
  LightGBM threshold: 0.95
Step 1: Processing 188 videos via cross-drive safe hardlink layer...

Processing 103_203_12_1_20250113_152455_doughnut_board_clueGiver_cam02.mp4 with COMBINED model...
Extracting features and model inferencing...


In [ ]:
from envisionhgdetector import utils

print("Step 2: Cutting segments...")
segments = utils.cut_video_by_segments(outputfolder)

# Check the gesture segments folder
gesture_segments_folder = os.path.join(outputfolder, "gesture_segments")
if os.path.exists(gesture_segments_folder):
    segment_files = [f for f in os.listdir(gesture_segments_folder) if f.endswith('.mp4')]
    print(f"Found {len(segment_files)} gesture segment files")

In [ ]:
# Create paths for analysis
gesture_segments_folder = os.path.join(outputfolder, "gesture_segments")
retracked_folder = os.path.join(outputfolder, "retracked")
analysis_folder = os.path.join(outputfolder, "analysis")

print("Step 4: Retracking gestures with world landmarks...")
tracking_results = detector.retrack_gestures(
    input_folder=gesture_segments_folder,
    output_folder=retracked_folder
)
print(f"Tracking results: {tracking_results}")

In [ ]:
if "error" not in tracking_results:
    print("Step 5: Computing DTW and kinematics...")
    analysis_results = detector.analyze_dtw_kinematics(
        landmarks_folder=tracking_results["landmarks_folder"],
        output_folder=analysis_folder
    )
    print(f"Analysis results: {analysis_results}")

In [ ]:
if "error" not in analysis_results:
    print("Step 6: Preparing dashboard...")
    detector.prepare_gesture_dashboard(
        data_folder=analysis_folder
    )